# Stage 05c: Production Model Training

**Purpose:** Train final model with best hyperparameters from 05b

**Inputs:**
- data/04c_train_encoded.parquet
- data/04c_test_encoded.parquet
- results/05b_best_params.yaml

**Outputs:**
- models/05c_model.ubj
- results/05c_predictions_train.parquet
- results/05c_predictions_test.parquet
- results/05c_metrics.yaml

In [ ]:
config_path = "config/car_coll/v1"

In [ ]:
import matplotlib
matplotlib.use("Agg")

import pandas as pd
import numpy as np
import xgboost as xgb
import yaml, os, sys
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error

sys.path.insert(0, str(Path.cwd() / "lib"))
from utils import setup_notebook_environment

print("########################################")
print("# STAGE 05c: PRODUCTION MODEL: INITIAL MODEL TRAINING")
print("########################################")

project_root = setup_notebook_environment()

In [ ]:
print(f"Python: {sys.version}")
print(f"XGBoost: {xgb.__version__}")

In [ ]:
# Get machine config
pc_num = open("current.pc").read().strip()
pc_id = f"PC{pc_num}"

config_file = f"{config_path}/config.yaml"
with open(config_file, "r") as f:
    cfg = yaml.safe_load(f)

output_base = cfg["machines"][pc_id]["paths"]["output_path"]
target = cfg["experiment"]["target"]
exposure = cfg["experiment"]["exposure"]

In [ ]:
# Load encoded features (already filtered by exclusions in 04c)
print(f"\n* Loading encoded features...")
X_train = pd.read_parquet(f"{output_base}/data/04c_train_encoded.parquet")
X_test = pd.read_parquet(f"{output_base}/data/04c_test_encoded.parquet")

# Load original data for target/exposure
train_orig = pd.read_parquet(f"{output_base}/data/04b_train.parquet")
test_orig = pd.read_parquet(f"{output_base}/data/04b_test.parquet")

y_train = train_orig[target]
y_test = test_orig[target]
w_train = train_orig[exposure]
w_test = test_orig[exposure]

print(f"  Train: {X_train.shape}")
print(f"  Test: {X_test.shape}")
print(f"  Features: {X_train.shape[1]}")

In [ ]:
# Load base_margin if available
base_margin_file_train = f"{output_base}/data/04c_train_base_margin.parquet"
base_margin_file_test = f"{output_base}/data/04c_test_base_margin.parquet"

if os.path.exists(base_margin_file_train):
    print(f"\n* Loading base_margin...")
    base_margin_train = pd.read_parquet(base_margin_file_train)["base_margin"].values
    base_margin_test = pd.read_parquet(base_margin_file_test)["base_margin"].values
    print(f"  Train base_margin: mean={base_margin_train.mean():.4f}")
    print(f"  Test base_margin: mean={base_margin_test.mean():.4f}")
else:
    base_margin_train = None
    base_margin_test = None
    print(f"\n* No base_margin found (GLM init disabled)")

In [ ]:
# Load monotonicity constraints
mono_file = f"{output_base}/config_generated/04c_monotonicity_constraints.yaml"
if os.path.exists(mono_file):
    print(f"\n* Loading monotonicity constraints...")
    with open(mono_file, "r") as f:
        mono_dict = yaml.safe_load(f)
    # Build constraints tuple in feature order
    feature_names = X_train.columns.tolist()
    monotone_constraints = tuple(mono_dict.get(f, 0) for f in feature_names)
    n_constrained = sum(1 for c in monotone_constraints if c != 0)
    print(f"  Loaded {n_constrained} constraints")
else:
    monotone_constraints = None
    print(f"\n* No monotonicity constraints found")

In [ ]:
# Load best hyperparameters from 05b
best_params_file = f'{output_base}/results/05b_best_params.yaml'
if os.path.exists(best_params_file):
    print(f'\n* Loading best params from 05b...')
    with open(best_params_file, 'r') as f:
        best_params = yaml.safe_load(f)
    print(f'  n_estimators: {best_params.get("n_estimators")}')
    print(f'  max_depth: {best_params.get("max_depth")}')
    print(f'  learning_rate: {best_params.get("learning_rate")}')
else:
    raise FileNotFoundError('Best params not found. Run stage 05b first.')

In [ ]:
# Build XGBoost parameters
print(f"\n* Building XGBoost parameters...")
xgb_params = cfg["xgboost"].copy()
# Override with best params from HPO
xgb_params.update(best_params)
n_estimators = xgb_params.pop("n_estimators")

# Add monotonicity constraints if available
if monotone_constraints:
    xgb_params["monotone_constraints"] = monotone_constraints

# Fix eval_metric for tweedie
if "eval_metric" in xgb_params and "tweedie" in xgb_params["eval_metric"]:
    if "@" not in xgb_params["eval_metric"]:
        variance_power = xgb_params["tweedie_variance_power"]
        xgb_params["eval_metric"] = f"{xgb_params['eval_metric']}@{variance_power}"

print(f"  n_estimators: {n_estimators}")
print(f"  monotone_constraints: {monotone_constraints is not None}")

In [ ]:
# Train model
print(f"\n* Training XGBoost model...")

model = xgb.XGBRegressor(n_estimators=n_estimators, **xgb_params)

model.fit(
    X_train, y_train,
    sample_weight=w_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    sample_weight_eval_set=[w_train, w_test],
    verbose=100
)

print(f"\n* Training complete")
print(f"  Best iteration: {model.best_iteration}")

In [ ]:
# Generate predictions
print(f"\n* Generating predictions...")
pred_train = model.predict(X_train)
pred_test = model.predict(X_test)

# Calculate metrics
mae_train = mean_absolute_error(y_train, pred_train, sample_weight=w_train)
mae_test = mean_absolute_error(y_test, pred_test, sample_weight=w_test)
rmse_train = np.sqrt(mean_squared_error(y_train, pred_train, sample_weight=w_train))
rmse_test = np.sqrt(mean_squared_error(y_test, pred_test, sample_weight=w_test))

print(f"\n* Metrics:")
print(f"  Train MAE: {mae_train:.4f}, RMSE: {rmse_train:.4f}")
print(f"  Test MAE: {mae_test:.4f}, RMSE: {rmse_test:.4f}")

In [ ]:
# Prepare data for lift charts
import matplotlib.pyplot as plt
from lift_chart_fast import create_lift_chart
from hpo_metrics import compute_fit_quality, compute_model_power
from IPython.display import Image, display

print(f'\n* Preparing lift chart data...')

# Add predictions to original data
train_orig['pred'] = pred_train
test_orig['pred'] = pred_test

# Create columns for lift_chart_fast
# denom = 1 because target (pp_coll) is already pure premium (rate)
train_orig['incurred_act'] = y_train.values
train_orig['incurred_pred'] = pred_train
train_orig['denom'] = 1

test_orig['incurred_act'] = y_test.values
test_orig['incurred_pred'] = pred_test
test_orig['denom'] = 1

print(f'  Data prepared (denom=1 for pure premium target)')

In [ ]:
# Train lift chart
print(f'\n* Creating train lift chart...')
fig_train, table_train = create_lift_chart(
    train_orig, 
    exposure,
    bins=10, 
    title='05c Production Train Lift Chart'
)

chart_file = f'{output_base}/results/05c_lift_train.png'
fig_train.savefig(chart_file, dpi=150, bbox_inches='tight')
plt.close(fig_train)
print(f'  Saved: {chart_file}')

display(Image(chart_file))
print('\nTrain decile summary:')
print(table_train[['decile', 'act', 'pred', 'act_rel', 'pred_rel']].to_string(index=False))

In [ ]:
# Test lift chart
print(f'\n* Creating test lift chart...')
fig_test, table_test = create_lift_chart(
    test_orig, 
    exposure,
    bins=10, 
    title='05c Production Test Lift Chart'
)

chart_file = f'{output_base}/results/05c_lift_test.png'
fig_test.savefig(chart_file, dpi=150, bbox_inches='tight')
plt.close(fig_test)
print(f'  Saved: {chart_file}')

display(Image(chart_file))
print('\nTest decile summary:')
print(table_test[['decile', 'act', 'pred', 'act_rel', 'pred_rel']].to_string(index=False))

In [ ]:
# Calculate actuarial metrics
print(f'\n* Calculating actuarial metrics...')

fit_train = compute_fit_quality(y_train, pred_train, w_train)
fit_test = compute_fit_quality(y_test, pred_test, w_test)
power_train = compute_model_power(y_train, pred_train, w_train)
power_test = compute_model_power(y_test, pred_test, w_test)

print(f'\n* Actuarial Metrics:')
print(f'  Train:')
print(f'    Fit Quality:  {fit_train:.4f}')
print(f'    Model Power:  {power_train:.4f}')
print(f'  Test:')
print(f'    Fit Quality:  {fit_test:.4f}')
print(f'    Model Power:  {power_test:.4f}')

In [ ]:
# Save model
os.makedirs(f"{output_base}/models", exist_ok=True)
model_file = f"{output_base}/models/05c_model.json"
model.get_booster().save_model(model_file)
print(f"\n* Saved model: {model_file}")

In [ ]:
# Save predictions
os.makedirs(f"{output_base}/results", exist_ok=True)

pd.DataFrame({
    "actual": y_train,
    "pred": pred_train,
    "exposure": w_train
}).to_parquet(f"{output_base}/results/05c_predictions_train.parquet", index=True)

pd.DataFrame({
    "actual": y_test,
    "pred": pred_test,
    "exposure": w_test
}).to_parquet(f"{output_base}/results/05c_predictions_test.parquet", index=True)

print(f"* Saved predictions")

In [ ]:
# Save metrics
metrics = {
    "stage": "05a_initial",
    "train": {"mae": float(mae_train), "rmse": float(rmse_train)},
    "test": {"mae": float(mae_test), "rmse": float(rmse_test)},
    "best_iteration": int(model.best_iteration),
    "n_features": int(X_train.shape[1]),
    "monotone_constraints_applied": monotone_constraints is not None,
    "base_margin_applied": base_margin_train is not None
}

with open(f"{output_base}/results/05c_metrics.yaml", "w") as f:
    yaml.dump(metrics, f)

print(f"* Saved metrics")

In [ ]:
print("\n########################################")
print("# STAGE 05a: COMPLETE")
print("########################################")